## Librerias y montaje de Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt


## Carga Inicial y Limpieza

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Ciencia de Datos e IA/Laboratorio de Datos II/Dataset/data/raw/customer_churn_historical.csv')

RANDOM_STATE = 42  # semilla fija para reproducibilidad

df.head() # cómo se ven los datos


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,6684-LWVH,Male,0,No,No,4,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,27.56,107.47,No
1,7027-JIFO,Male,0,No,No,35,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,30.52,1020.47,No
2,5981-VOQO,Female,0,No,No,45,Yes,Yes,DSL,No,...,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),66.87,2686.35,No
3,7266-HYDM,Male,0,Yes,No,31,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),86.95,2536.96,Yes
4,2821-JDLS,Female,0,Yes,No,5,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,30.18,139.12,No


In [ ]:
df = df.drop(columns=["customerID"]) # dropeo de ID, no uitlizable en el modelo

In [ ]:
# Conversion de MonthlyCHarges y TotalChagers a numérico
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

## EDA

In [ ]:
df.shape # cuántas filas y columnas

(7043, 20)

In [ ]:
df.info()  # tipos datos y cantidad de no nulos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [ ]:
df.describe() # rangos y órdenes de magnitud

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7017.000000
mean,0.170098,35.168394,68.168503,2312.077586
std,0.375746,18.901478,24.980659,1573.967858
min,0.000000,0.000000,18.000000,0.000000
25%,0.000000,20.000000,55.360000,986.630000
50%,0.000000,35.000000,73.910000,2013.200000
75%,0.000000,51.000000,88.000000,3452.850000
max,1.000000,72.000000,114.410000,7761.340000


In [ ]:
# Tratamiento de nulos en todas las columnas

print("Valores nulos por columna:")
print(df.isnull().sum())

# for col in df.columns:
#    if df[col].isnull().sum() > 0:          # solo si esa columna tiene nulos
#        if df[col].dtype in ["float64", "int64"]:
#            df[col] = df[col].fillna(df[col].median())   # numéricas -> mediana
#        else:
#            df[col] = df[col].fillna(df[col].mode()[0])  # categóricas -> moda



Valores nulos por columna:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        26
Churn                0
dtype: int64


## Analisis de correlaciones con la variable Churn

In [ ]:
# Correlacion con variables numericas (cercano a 1 o -1 indica evidencia de asociación)

df["Churn_num"] = df["Churn"].map({"Yes": 1, "No": 0}) #conversion de Churn a 1/0

correlaciones = df.corr(numeric_only=True)["Churn_num"].sort_values(ascending=False)
print(correlaciones)

Churn_num         1.000000
MonthlyCharges    0.282891
SeniorCitizen     0.036132
TotalCharges     -0.013405
tenure           -0.186551
Name: Churn_num, dtype: float64


In [ ]:
# Análisis de asociación entre variables categóricas y Churn mediante chi2

from scipy.stats import chi2_contingency

for col in df.select_dtypes(include="object").columns:
    if col != "Churn":
        tabla = pd.crosstab(df[col], df["Churn"])
        chi2, p, dof, expected = chi2_contingency(tabla)
        print(f"{col}: p-valor = {p:.5f}")

gender: p-valor = 0.45381
Partner: p-valor = 0.00397
Dependents: p-valor = 0.01036
PhoneService: p-valor = 0.18360
MultipleLines: p-valor = 0.14764
InternetService: p-valor = 0.00000
OnlineSecurity: p-valor = 0.00000
OnlineBackup: p-valor = 0.00000
DeviceProtection: p-valor = 0.00000
TechSupport: p-valor = 0.00000
StreamingTV: p-valor = 0.00000
StreamingMovies: p-valor = 0.00000
Contract: p-valor = 0.00000
PaperlessBilling: p-valor = 0.00806
PaymentMethod: p-valor = 0.00000


## Selección de variables

A partir del análisis realizado, se seleccionan inicialmente las variables:

- Partner
- Dependents
- PaperlessBilling

Estas variables serán utilizadas en la primera experimentación del modelo.

Como alternativas para posteriores experimentos se consideran:
- MonthlyCharges
- tenure
- Contract
- InternetService

In [ ]:
features = [
    "Partner",
    "Dependents",
    "PaperlessBilling"
]

## Separación Train y Test


In [ ]:
X = df[features]
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 3)
X_test : (1409, 3)
y_train: (5634,)
y_test : (1409,)


## Preprocessing

In [ ]:
# Al tener variables categóricas podemos utilizar OneHotEncoder para trasnformar las variables categóricas en numéricas
# Se utiliza SimpleImputer para dejar la imputación dentro del pipeline
# Se incorpora "handle_unknown = ignore" para no tratar la aparición de nuevas categorías

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
# Se agrega esta opción por si luego se incorporan varibales numéricas
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, features)
    ]
)

## Pipeline

In [ ]:
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

## Entrenamiento

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

## Predicciones

In [ ]:
y_pred = model.predict(X_test)

## Obtener Probabilidades de Churn

In [ ]:
model.named_steps["classifier"].classes_ # Clases que reconoce el árbol

array(['No', 'Yes'], dtype=object)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]  # Tomamos la probabilidad correspondiente a la segunda clase --> Churn = "Yes"


In [ ]:
# Comprobaciones

print("Primeras 10 predicciones:")
print(y_pred[:10])

print("\nPrimeras 10 probabilidades de churn:")
print(y_prob[:10])

Primeras 10 predicciones:
['No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'No']

Primeras 10 probabilidades de churn:
[0.25094578 0.25094578 0.25094578 0.25957447 0.24900398 0.25094578
 0.26875    0.26875    0.26875    0.25707547]


## Evaluación

In [ ]:
precision = precision_score(y_test, y_pred, pos_label="Yes")
recall = recall_score(y_test, y_pred, pos_label="Yes")
f1 = f1_score(y_test, y_pred, pos_label="Yes")
roc_auc = roc_auc_score(
    (y_test == "Yes").astype(int),
    y_prob
)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("Roc-Auc:", roc_auc)

Precision: 0.0
Recall: 0.0
F1-Score: 0.0
Roc-Auc: 0.5499774473512302


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

Matriz de confusión:
[[1037    0]
 [ 372    0]]


## Análisis Primer Experimento


| Métrica   | Resultado | Interpretación                              |
| --------- | --------: | ------------------------------------------- |
| Precision |      0.00 | No hizo predicciones positivas              |
| Recall    |      0.00 | No detectó ningún churn                     |
| F1-Score        |      0.00 | Muy mal equilibrio entre precisión y recall |
| ROC-AUC   |      0.55 | Separación muy débil, apenas sobre azar     |
| Falsos Negativos        |       372 | **372 churn reales no detectados**          |


Features: Partner; Dependents; PaperlessBilling

Decision Tree: el modelo no logró detectar la clase Yes con las variables seleccionadas y obtuvo un ROC-AUC cercano a 0.55.

## Experimento 2 - Baseline

In [ ]:
baseline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DummyClassifier(
        strategy="most_frequent"
    ))
])

In [ ]:
baseline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling'])])),
                ('classifier', DummyClassifier(strategy='most_frequent'))])

In [ ]:
y_pred_baseline = baseline.predict(X_test)

In [ ]:
y_prob_baseline = baseline.predict_proba(X_test)[:, 1]

In [ ]:
print("Primeras 10 predicciones:")
print(y_pred_baseline[:10])

print("\nPrimeras 10 probabilidades de churn:")
print(y_prob_baseline[:10])

Primeras 10 predicciones:
['No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'No']

Primeras 10 probabilidades de churn:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [ ]:
precision_baseline = precision_score(
    y_test,
    y_pred_baseline,
    pos_label="Yes"
)

recall_baseline = recall_score(
    y_test,
    y_pred_baseline,
    pos_label="Yes"
)

f1_baseline = f1_score(
    y_test,
    y_pred_baseline,
    pos_label="Yes"
)

roc_auc_baseline = roc_auc_score(
    (y_test == "Yes").astype(int),
    y_prob_baseline
)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("Precision:", precision_baseline)
print("Recall:", recall_baseline)
print("F1-Score:", f1_baseline)
print("ROC-AUC:", roc_auc_baseline)

Precision: 0.0
Recall: 0.0
F1-Score: 0.0
ROC-AUC: 0.5


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_baseline))

Matriz de confusión:
[[1037    0]
 [ 372    0]]


## Análisis Segundo Experimento

| Métrica   | Resultado | Interpretación                              |
| --------- | --------: | ------------------------------------------- |
| Precision |      0.00 | No hizo predicciones positivas              |
| Recall    |      0.00 | No detectó ningún churn                     |
| F1-Score        |      0.00 | Muy mal equilibrio entre precisión y recall |
| ROC-AUC   |      0.50 | Separación muy débil   |
| Falsos Negativos        |       372 | **372 churn reales no detectados**          |


DummyClassifier: el baseline predice siempre la clase mayoritaria y obtiene ROC-AUC = 0.50.

Comparación vs. Experimento 1: el árbol muestra una mejora muy pequeña respecto del baseline en capacidad de discriminación, pero su comportamiento de clasificación sigue siendo equivalente al baseline, con Recall y F1 iguales a 0.

## Experimento 3 - Decision Tree con TotalCharges

In [ ]:
categorical_features_exp3 = [
    "Partner",
    "Dependents",
    "PaperlessBilling"
]

numeric_features_exp3 = [
    "TotalCharges"
]

features_exp3 = categorical_features_exp3 + numeric_features_exp3

In [ ]:
# Nueva variable "X" para este experimento 3 así evitamos pisar lo realizado antes
X_exp3 = df[features_exp3]

In [ ]:
# Utilizamos exactamente las mismas filas que estuvieron en train y test para que sea una comparativa idéntica

X_train_exp3 = X_exp3.loc[X_train.index]
X_test_exp3 = X_exp3.loc[X_test.index]

y_train_exp3 = y_train.copy()
y_test_exp3 = y_test.copy()

In [ ]:
print("Nulos de TotalCharges en Train:", X_train_exp3["TotalCharges"].isnull().sum())
print("Nulos de TotalCharges en Test :", X_test_exp3["TotalCharges"].isnull().sum())

Nulos de TotalCharges en Train: 19
Nulos de TotalCharges en Test : 7


In [ ]:
# Pipeline Categórico y Numérico

categorical_pipeline_exp3 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline_exp3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
# Construimos el nuevo Column Transformer

preprocessor_exp3 = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline_exp3, categorical_features_exp3),
        ("numeric", numeric_pipeline_exp3, numeric_features_exp3)
    ]
)

In [ ]:
# Nuevo Pipeline Completo

model_exp3 = Pipeline([
    ("preprocessing", preprocessor_exp3),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

In [ ]:
# Entrenamos

model_exp3.fit(X_train_exp3, y_train_exp3)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [ ]:
# Predecimos y obtenemos probabilidades

y_pred_exp3 = model_exp3.predict(X_test_exp3)
y_prob_exp3 = model_exp3.predict_proba(X_test_exp3)[:, 1]

In [ ]:
# Comprobamos clases

model_exp3.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
# Evaluamos con mismas métricas que experimento anterior

precision_exp3 = precision_score(
    y_test_exp3,
    y_pred_exp3,
    pos_label="Yes"
)

recall_exp3 = recall_score(
    y_test_exp3,
    y_pred_exp3,
    pos_label="Yes"
)

f1_exp3 = f1_score(
    y_test_exp3,
    y_pred_exp3,
    pos_label="Yes"
)

roc_auc_exp3 = roc_auc_score(
    (y_test_exp3 == "Yes").astype(int),
    y_prob_exp3
)

In [ ]:
print("Precision:", precision_exp3)
print("Recall:", recall_exp3)
print("F1-Score:", f1_exp3)
print("ROC-AUC:", roc_auc_exp3)

Precision: 0.3
Recall: 0.2903225806451613
F1-Score: 0.29508196721311475
ROC-AUC: 0.5236517663649277


In [ ]:
# Matriz de confusión

print("Matriz de confusión:")
print(confusion_matrix(y_test_exp3, y_pred_exp3))

Matriz de confusión:
[[785 252]
 [264 108]]



*  TN = 785 → clasificamos correctamente 785 clientes como No.
*  FP = 252 → marcamos como churn a 252 clientes que no abandonaron.
*  FN = 264 → dejamos pasar 264 churn reales.
*  TP = 108 → detectamos 108 churn reales.










### Análisis del Experimento 3

Al incorporar `TotalCharges` como variable predictora y tratar sus valores faltantes
mediante `SimpleImputer` dentro del pipeline, el modelo comienza a identificar
casos de churn.

Resultado:

- Precision: 0.30
- Recall: 0.29
- F1-Score: 0.30
- ROC-AUC: 0.52
- Falsos Negativos: 264

Respecto de experimentos anteriores, el modelo logra detectar casos de la clase "Yes",
aunque mantiene una cantidad significativa de falsos negativos y una capacidad
de discriminación global cercana a lo aleatorio.

## Experimento 4 - Logistic Regression

In [ ]:
# Pipeline

logistic_model = Pipeline([
    ("preprocessing", preprocessor_exp3),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

In [ ]:
# Entrenamos

logistic_model.fit(X_train_exp3, y_train_exp3)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [ ]:
# Comprobamos clases

logistic_model.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
# Predicción y probabilidad

y_pred_logistic = logistic_model.predict(X_test_exp3)
y_prob_logistic = logistic_model.predict_proba(X_test_exp3)[:, 1]

In [ ]:
# Evaluación

precision_logistic = precision_score(
    y_test_exp3,
    y_pred_logistic,
    pos_label="Yes"
)

recall_logistic = recall_score(
    y_test_exp3,
    y_pred_logistic,
    pos_label="Yes"
)

f1_logistic = f1_score(
    y_test_exp3,
    y_pred_logistic,
    pos_label="Yes"
)

roc_auc_logistic = roc_auc_score(
    (y_test_exp3 == "Yes").astype(int),
    y_prob_logistic
)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("Precision:", precision_logistic)
print("Recall:", recall_logistic)
print("F1-Score:", f1_logistic)
print("ROC-AUC:", roc_auc_logistic)

Precision: 0.0
Recall: 0.0
F1-Score: 0.0
ROC-AUC: 0.5399959560767723


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test_exp3, y_pred_logistic))

Matriz de confusión:
[[1037    0]
 [ 372    0]]


### Análisis del Experimento 4

La regresión logística, utilizando las mismas variables y el mismo preprocesamiento que el Experimento 3, no realizó predicciones de la clase "Yes" sobre el conjunto de Test.

Resultado:

- Precision: 0.00
- Recall: 0.00
- F1-Score: 0.00
- ROC-AUC: 0.54
- Falsos Negativos: 372

El ROC-AUC se encuentra ligeramente por encima del baseline de 0.50,
pero la clasificación final no logra detectar casos de churn.

# Experimento 5 - Selección y evaluación de nuevas features

En esta nueva etapa experimental continuamos utilizando el chi-cuadrado como evidencia de asociación pero además medimos la diferencia en las tasas de churn entre las categorías para identificar nuevas variables candidatas.

In [ ]:
# Análisis Categórico con tabla ordenada por diferencia de churn;

categorical_analysis = []

for col in df.select_dtypes(include="object").columns:
    if col != "Churn":

        tabla = pd.crosstab(df[col], df["Churn"])

        # Prueba chi-cuadrado
        chi2, p, dof, expected = chi2_contingency(tabla)

        # Tasa de churn por categoría
        churn_rate = df.groupby(col)["Churn_num"].mean()

        categorical_analysis.append({
            "Variable": col,
            "p_value": p,
            "churn_min": churn_rate.min(),
            "churn_max": churn_rate.max(),
            "diferencia_churn": churn_rate.max() - churn_rate.min()
        })

categorical_analysis = pd.DataFrame(categorical_analysis)

categorical_analysis.sort_values(
    "diferencia_churn",
    ascending=False
)

,Variable,p_value,churn_min,churn_max,diferencia_churn
5,InternetService,1.467974e-140,0.076178,0.402528,0.326349
12,Contract,6.031149e-146,0.089744,0.387300,0.297556
6,OnlineSecurity,8.903468e-89,0.076178,0.350609,0.274431
9,TechSupport,9.705530e-85,0.076178,0.341795,0.265617
7,OnlineBackup,2.444422e-80,0.076178,0.334293,0.258115
8,DeviceProtection,6.717717e-80,0.076178,0.331313,0.255135
10,StreamingTV,8.779265e-80,0.076178,0.329295,0.253117
11,StreamingMovies,4.261268e-79,0.076178,0.323605,0.247427
14,PaymentMethod,3.940796e-09,0.230203,0.310562,0.080359
4,MultipleLines,1.476367e-01,0.241481,0.274382,0.032900


In [ ]:
# Ordenado por p-value y evidencia estadística

categorical_analysis.sort_values(
    ["p_value", "diferencia_churn"],
    ascending=[True, False]
)

,Variable,p_value,churn_min,churn_max,diferencia_churn
12,Contract,6.031149e-146,0.089744,0.387300,0.297556
5,InternetService,1.467974e-140,0.076178,0.402528,0.326349
6,OnlineSecurity,8.903468e-89,0.076178,0.350609,0.274431
9,TechSupport,9.705530e-85,0.076178,0.341795,0.265617
7,OnlineBackup,2.444422e-80,0.076178,0.334293,0.258115
8,DeviceProtection,6.717717e-80,0.076178,0.331313,0.255135
10,StreamingTV,8.779265e-80,0.076178,0.329295,0.253117
11,StreamingMovies,4.261268e-79,0.076178,0.323605,0.247427
14,PaymentMethod,3.940796e-09,0.230203,0.310562,0.080359
1,Partner,3.968770e-03,0.247953,0.278498,0.030545


In [ ]:
# Análisis Variables Numéricas

numeric_analysis = pd.DataFrame({
    "Variable": correlaciones.index,
    "correlacion": correlaciones.values
})

numeric_analysis = numeric_analysis[
    numeric_analysis["Variable"] != "Churn_num"
]

numeric_analysis["abs_correlacion"] = (
    numeric_analysis["correlacion"].abs()
)

numeric_analysis.sort_values(
    "abs_correlacion",
    ascending=False
)

,Variable,correlacion,abs_correlacion
1,MonthlyCharges,0.282891,0.282891
4,tenure,-0.186551,0.186551
2,SeniorCitizen,0.036132,0.036132
3,TotalCharges,-0.013405,0.013405


### Selección de variables candidatas

A partir del nuevo análisis exploratorio se identificaron como candidatas
las variables `MonthlyCharges` y `tenure` entre las numéricas,
por presentar las mayores magnitudes de correlación con `Churn`.

Entre las variables categóricas se identificaron principalmente
`InternetService` y `Contract`, por la evidencia obtenida mediante chi-cuadrado y por las mayores diferencias observadas en las tasas de churn entre sus categorías.

Estas variables serán incorporadas progresivamente a nuevos
experimentos para evaluar su aporte.

El resto de las variables y la partición train/test se mantienen sin cambios.

In [ ]:
# Incorporamos MonthlyCharges

categorical_features_exp5 = [
    "Partner",
    "Dependents",
    "PaperlessBilling"
]

numeric_features_exp5 = [
    "TotalCharges",
    "MonthlyCharges"
]

features_exp5 = categorical_features_exp5 + numeric_features_exp5

In [ ]:
X_exp5 = df[features_exp5]

In [ ]:
X_train_exp5 = X_exp5.loc[X_train_exp3.index]
X_test_exp5 = X_exp5.loc[X_test_exp3.index]

y_train_exp5 = y_train_exp3.copy()
y_test_exp5 = y_test_exp3.copy()

In [ ]:
# Revisamos que MonthlyCharges esté agregada

print("Train:")
print(X_train_exp5[features_exp5].head())

print("\nTest:")
print(X_test_exp5[features_exp5].head())

Train:
     Partner Dependents PaperlessBilling  TotalCharges  MonthlyCharges
263      Yes        Yes              Yes       1017.34           79.82
6008     Yes        Yes              Yes       1196.92           23.77
4246     Yes         No              Yes       5013.92           94.21
6981      No         No               No        855.79           28.68
4489     Yes        Yes               No       2344.93           59.84

Test:
     Partner Dependents PaperlessBilling  TotalCharges  MonthlyCharges
450      Yes         No               No       1665.53           35.15
695      Yes         No               No       1306.39           30.92
2229     Yes         No               No       5680.60           95.08
4420     Yes        Yes              Yes        745.95           58.93
6870      No        Yes              Yes        978.14           29.58


In [ ]:
# Pipeline Categórico y Numérico

categorical_pipeline_exp5 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline_exp5 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
# Creamos el ColumnTransformer

preprocessor_exp5 = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline_exp5, categorical_features_exp5),
        ("numeric", numeric_pipeline_exp5, numeric_features_exp5)
    ]
)

In [ ]:
# Creamos el modelo

model_exp5 = Pipeline([
    ("preprocessing", preprocessor_exp5),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

In [ ]:
# Entrenamiento

model_exp5.fit(
    X_train_exp5,
    y_train_exp5
)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges',
                                                   'MonthlyCharges'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [ ]:
# Predicción y probabilidad + Comprob. de clases

y_pred_exp5 = model_exp5.predict(X_test_exp5)

y_prob_exp5 = model_exp5.predict_proba(X_test_exp5)[:, 1]

model_exp5.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
# Evaluación

precision_exp5 = precision_score(
    y_test_exp5,
    y_pred_exp5,
    pos_label="Yes"
)

recall_exp5 = recall_score(
    y_test_exp5,
    y_pred_exp5,
    pos_label="Yes"
)

f1_exp5 = f1_score(
    y_test_exp5,
    y_pred_exp5,
    pos_label="Yes"
)

roc_auc_exp5 = roc_auc_score(
    (y_test_exp5 == "Yes").astype(int),
    y_prob_exp5
)

In [ ]:
print("Precision:", precision_exp5)
print("Recall:", recall_exp5)
print("F1-Score:", f1_exp5)
print("ROC-AUC:", roc_auc_exp5)

Precision: 0.4197860962566845
Recall: 0.4220430107526882
F1-Score: 0.42091152815013405
ROC-AUC: 0.6063927686357462


In [ ]:
# Matriz de Confusión

print("Matriz de confusión:")
print(confusion_matrix(y_test_exp5, y_pred_exp5))

Matriz de confusión:
[[820 217]
 [215 157]]


Evolución en la Matriz de Confusión vs. Experimento 3

*   TN: 785 → 820
*   FP: 252 → 217
*   FN: 264 → 215
*   TP: 108 → 157







Respecto del Experimento 3, se observa una mejora en Precision, Recall,
F1-Score y ROC-AUC. Además, los falsos negativos disminuyen de 264 a 215 y aumentan los negativos reales.

No solo mejoró una métrica aislada, mejoraron las cuatro métricas que estamos utilizando para evaluar este experimento.

Estos resultados proporcionan evidencia experimental de que
`MonthlyCharges` aporta información adicional al modelo.



## Experimento 6 - Incorporación de "tenure"

In [ ]:
categorical_features_exp6 = [
    "Partner",
    "Dependents",
    "PaperlessBilling"
]

numeric_features_exp6 = [
    "TotalCharges",
    "MonthlyCharges",
    "tenure"
]

features_exp6 = categorical_features_exp6 + numeric_features_exp6

In [ ]:
X_exp6 = df[features_exp6]

In [ ]:
X_train_exp6 = X_exp6.loc[X_train_exp5.index]
X_test_exp6 = X_exp6.loc[X_test_exp5.index]

y_train_exp6 = y_train_exp5.copy()
y_test_exp6 = y_test_exp5.copy()

In [ ]:
# Al ser variable numérica queremos comprobar que no tenga nulos

print("Nulos de tenure en Train:",
      X_train_exp6["tenure"].isnull().sum())

print("Nulos de tenure en Test:",
      X_test_exp6["tenure"].isnull().sum())

Nulos de tenure en Train: 0
Nulos de tenure en Test: 0


In [ ]:
# Pipeline categórico y numérico

categorical_pipeline_exp6 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline_exp6 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
preprocessor_exp6 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_pipeline_exp6,
            categorical_features_exp6
        ),
        (
            "numeric",
            numeric_pipeline_exp6,
            numeric_features_exp6
        )
    ]
)

In [ ]:
# Creamos el modelo

model_exp6 = Pipeline([
    ("preprocessing", preprocessor_exp6),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

In [ ]:
# Entrenamos

model_exp6.fit(
    X_train_exp6,
    y_train_exp6
)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges',
                                                   'MonthlyCharges',
                                                   'tenure'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [ ]:
# Comprobamos clases

model_exp6.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
# Predicción y probabilidad

y_pred_exp6 = model_exp6.predict(X_test_exp6)

y_prob_exp6 = model_exp6.predict_proba(X_test_exp6)[:, 1]

In [ ]:
# Métricas (iguales a los experimentos anteriores)

precision_exp6 = precision_score(
    y_test_exp6,
    y_pred_exp6,
    pos_label="Yes"
)

recall_exp6 = recall_score(
    y_test_exp6,
    y_pred_exp6,
    pos_label="Yes"
)

f1_exp6 = f1_score(
    y_test_exp6,
    y_pred_exp6,
    pos_label="Yes"
)

roc_auc_exp6 = roc_auc_score(
    (y_test_exp6 == "Yes").astype(int),
    y_prob_exp6
)

In [ ]:
print("Precision:", precision_exp6)
print("Recall:", recall_exp6)
print("F1-Score:", f1_exp6)
print("ROC-AUC:", roc_auc_exp6)

Precision: 0.35294117647058826
Recall: 0.3870967741935484
F1-Score: 0.36923076923076925
ROC-AUC: 0.5662581267303326


In [ ]:
# Matriz de confusión

print("Matriz de confusión:")
print(confusion_matrix(y_test_exp6, y_pred_exp6))

Matriz de confusión:
[[773 264]
 [228 144]]


Al agregar "tenure" las cuatro métricas bajaron respecto del Experimento 5.

Además también empeoró la matriz de confusión:

*   TP baja de 157 → 144
*   FN sube de 215 → 228
*   FP sube de 217 → 264
*   TN baja de 820 → 773

Es decir, en esta configuración tenure no aportó una mejora adicional respecto del conjunto que ya incluía MonthlyCharges.



## Experimento 7 - Incorporación de Contract

In [ ]:
categorical_features_exp7 = [
    "Partner",
    "Dependents",
    "PaperlessBilling",
    "Contract"
]

numeric_features_exp7 = [
    "TotalCharges",
    "MonthlyCharges"
]

features_exp7 = categorical_features_exp7 + numeric_features_exp7

In [ ]:
X_exp7 = df[features_exp7]

In [ ]:
X_train_exp7 = X_exp7.loc[X_train_exp5.index]
X_test_exp7 = X_exp7.loc[X_test_exp5.index]

y_train_exp7 = y_train_exp5.copy()
y_test_exp7 = y_test_exp5.copy()

In [ ]:
print("Train:", X_train_exp7.shape)
print("Test :", X_test_exp7.shape)

Train: (5634, 6)
Test : (1409, 6)


In [ ]:
print("\nPrimeras filas de Train:")
print(X_train_exp7.head())


Primeras filas de Train:
     Partner Dependents PaperlessBilling        Contract  TotalCharges  \
263      Yes        Yes              Yes        Two year       1017.34   
6008     Yes        Yes              Yes        One year       1196.92   
4246     Yes         No              Yes        One year       5013.92   
6981      No         No               No  Month-to-month        855.79   
4489     Yes        Yes               No        One year       2344.93   

      MonthlyCharges  
263            79.82  
6008           23.77  
4246           94.21  
6981           28.68  
4489           59.84  


In [ ]:
categorical_pipeline_exp7 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline_exp7 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
preprocessor_exp7 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_pipeline_exp7,
            categorical_features_exp7
        ),
        (
            "numeric",
            numeric_pipeline_exp7,
            numeric_features_exp7
        )
    ]
)

In [ ]:
model_exp7 = Pipeline([
    ("preprocessing", preprocessor_exp7),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

In [ ]:
model_exp7.fit(
    X_train_exp7,
    y_train_exp7
)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling',
                                                   'Contract']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges',
                                                   'MonthlyCharges'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [ ]:
y_pred_exp7 = model_exp7.predict(X_test_exp7)
y_prob_exp7 = model_exp7.predict_proba(X_test_exp7)[:, 1]

In [ ]:
model_exp7.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
precision_exp7 = precision_score(
    y_test_exp7,
    y_pred_exp7,
    pos_label="Yes"
)

recall_exp7 = recall_score(
    y_test_exp7,
    y_pred_exp7,
    pos_label="Yes"
)

f1_exp7 = f1_score(
    y_test_exp7,
    y_pred_exp7,
    pos_label="Yes"
)

roc_auc_exp7 = roc_auc_score(
    (y_test_exp7 == "Yes").astype(int),
    y_prob_exp7
)

In [ ]:
print("Precision:", precision_exp7)
print("Recall:", recall_exp7)
print("F1-Score:", f1_exp7)
print("ROC-AUC:", roc_auc_exp7)

Precision: 0.42134831460674155
Recall: 0.4032258064516129
F1-Score: 0.41208791208791207
ROC-AUC: 0.6022879273338104


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test_exp7, y_pred_exp7))

Matriz de confusión:
[[831 206]
 [222 150]]


Al incorporar `Contract` se observa una ligera mejora en Precision, pero
una disminución en Recall, F1-Score y ROC-AUC respecto del Experimento 5.

La matriz de confusión muestra que los falsos positivos disminuyen de
217 a 206, pero los falsos negativos aumentan de 215 a 222.

Por lo tanto, la incorporación de
`Contract` no mejora el desempeño del Experimento 5 pero tampoco empeora, se acerca bastante.

## Experimento 8 - Incorporación de InternetService

In [ ]:
categorical_features_exp8 = [
    "Partner",
    "Dependents",
    "PaperlessBilling",
    "InternetService"
]

numeric_features_exp8 = [
    "TotalCharges",
    "MonthlyCharges"
]

features_exp8 = categorical_features_exp8 + numeric_features_exp8

In [ ]:
X_exp8 = df[features_exp8]

In [ ]:
X_train_exp8 = X_exp8.loc[X_train_exp5.index]
X_test_exp8 = X_exp8.loc[X_test_exp5.index]

y_train_exp8 = y_train_exp5.copy()
y_test_exp8 = y_test_exp5.copy()

In [ ]:
print("Train:", X_train_exp8.shape)
print("Test :", X_test_exp8.shape)

print("\nPrimeras filas:")
print(X_train_exp8.head())

Train: (5634, 6)
Test : (1409, 6)

Primeras filas:
     Partner Dependents PaperlessBilling InternetService  TotalCharges  \
263      Yes        Yes              Yes     Fiber optic       1017.34   
6008     Yes        Yes              Yes              No       1196.92   
4246     Yes         No              Yes     Fiber optic       5013.92   
6981      No         No               No              No        855.79   
4489     Yes        Yes               No             DSL       2344.93   

      MonthlyCharges  
263            79.82  
6008           23.77  
4246           94.21  
6981           28.68  
4489           59.84  


In [ ]:
categorical_pipeline_exp8 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline_exp8 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
preprocessor_exp8 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_pipeline_exp8,
            categorical_features_exp8
        ),
        (
            "numeric",
            numeric_pipeline_exp8,
            numeric_features_exp8
        )
    ]
)

In [ ]:
model_exp8 = Pipeline([
    ("preprocessing", preprocessor_exp8),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

In [ ]:
model_exp8.fit(
    X_train_exp8,
    y_train_exp8
)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling',
                                                   'InternetService']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges',
                                                   'MonthlyCharges'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [ ]:
model_exp8.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
y_pred_exp8 = model_exp8.predict(X_test_exp8)

y_prob_exp8 = model_exp8.predict_proba(X_test_exp8)[:, 1]

In [ ]:
precision_exp8 = precision_score(
    y_test_exp8,
    y_pred_exp8,
    pos_label="Yes"
)

recall_exp8 = recall_score(
    y_test_exp8,
    y_pred_exp8,
    pos_label="Yes"
)

f1_exp8 = f1_score(
    y_test_exp8,
    y_pred_exp8,
    pos_label="Yes"
)

roc_auc_exp8 = roc_auc_score(
    (y_test_exp8 == "Yes").astype(int),
    y_prob_exp8
)

In [ ]:
print("Precision:", precision_exp8)
print("Recall:", recall_exp8)
print("F1-Score:", f1_exp8)
print("ROC-AUC:", roc_auc_exp8)

Precision: 0.3829787234042553
Recall: 0.3870967741935484
F1-Score: 0.3850267379679144
ROC-AUC: 0.5816872491989922


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test_exp8, y_pred_exp8))

Matriz de confusión:
[[805 232]
 [228 144]]


Respecto del Experimento 5, todas las métricas evaluadas disminuyen.
Además, los falsos negativos aumentan de 215 a 228.

La incorporación de `InternetService` no aporta una mejora respecto del Experimento 5.

## Experimento 9 - Reemplazo de PaperlessBilling por Contract

`Contract` presentó evidencia de asociación con `Churn` mediante la prueba
de chi-cuadrado y una diferencia relevante entre las tasas mínima y máxima
de churn observadas.

En este experimento se propone reemplazar `PaperlessBilling` por `Contract`
para evaluar si `Contract` aporta mayor información predictiva dentro de
un conjunto de features de igual tamaño.

In [ ]:
categorical_features_exp9 = [
    "Partner",
    "Dependents",
    "Contract"
]

numeric_features_exp9 = [
    "TotalCharges",
    "MonthlyCharges"
]

features_exp9 = categorical_features_exp9 + numeric_features_exp9

In [ ]:
X_exp9 = df[features_exp9]

In [ ]:
X_train_exp9 = X_exp9.loc[X_train_exp5.index]
X_test_exp9 = X_exp9.loc[X_test_exp5.index]

y_train_exp9 = y_train_exp5.copy()
y_test_exp9 = y_test_exp5.copy()

In [ ]:
print("Train:", X_train_exp9.shape)
print("Test :", X_test_exp9.shape)

print("\nFeatures utilizadas:")
print(features_exp9)

Train: (5634, 5)
Test : (1409, 5)

Features utilizadas:
['Partner', 'Dependents', 'Contract', 'TotalCharges', 'MonthlyCharges']


In [ ]:
categorical_pipeline_exp9 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline_exp9 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])


In [ ]:
preprocessor_exp9 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_pipeline_exp9,
            categorical_features_exp9
        ),
        (
            "numeric",
            numeric_pipeline_exp9,
            numeric_features_exp9
        )
    ]
)

In [ ]:
model_exp9 = Pipeline([
    ("preprocessing", preprocessor_exp9),
    ("classifier", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

In [ ]:
model_exp9.fit(
    X_train_exp9,
    y_train_exp9
)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'Contract']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges',
                                                   'MonthlyCharges'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [ ]:
model_exp9.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
y_pred_exp9 = model_exp9.predict(X_test_exp9)

y_prob_exp9 = model_exp9.predict_proba(X_test_exp9)[:, 1]

In [ ]:
precision_exp9 = precision_score(
    y_test_exp9,
    y_pred_exp9,
    pos_label="Yes"
)

recall_exp9 = recall_score(
    y_test_exp9,
    y_pred_exp9,
    pos_label="Yes"
)

f1_exp9 = f1_score(
    y_test_exp9,
    y_pred_exp9,
    pos_label="Yes"
)

roc_auc_exp9 = roc_auc_score(
    (y_test_exp9 == "Yes").astype(int),
    y_prob_exp9
)

In [ ]:
print("Precision:", precision_exp9)
print("Recall:", recall_exp9)
print("F1-Score:", f1_exp9)
print("ROC-AUC:", roc_auc_exp9)

Precision: 0.42
Recall: 0.3951612903225806
F1-Score: 0.407202216066482
ROC-AUC: 0.5997021495007311


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test_exp9, y_pred_exp9))

Matriz de confusión:
[[834 203]
 [225 147]]


## Experimento 10 - Random Forest

El Experimento 5 obtuvo hasta el momento el mejor desempeño de nuestra
secuencia de experimentación utilizando las variables:

- Partner
- Dependents
- PaperlessBilling
- TotalCharges
- MonthlyCharges

En este experimento se reemplaza el Decision Tree por un Random Forest,
manteniendo constantes las variables, la partición train/test,
el preprocessing y el random_state.

In [ ]:
model_exp10 = Pipeline([
    ("preprocessing", preprocessor_exp5),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE
    ))
])

In [ ]:
model_exp10.fit(
    X_train_exp5,
    y_train_exp5
)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Partner', 'Dependents',
                                                   'PaperlessBilling']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TotalCharges',
                                                   'MonthlyCharges'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [ ]:
model_exp10.named_steps["classifier"].classes_

array(['No', 'Yes'], dtype=object)

In [ ]:
y_pred_exp10 = model_exp10.predict(X_test_exp5)

y_prob_exp10 = model_exp10.predict_proba(X_test_exp5)[:, 1]

In [ ]:
precision_exp10 = precision_score(
    y_test_exp5,
    y_pred_exp10,
    pos_label="Yes"
)

recall_exp10 = recall_score(
    y_test_exp5,
    y_pred_exp10,
    pos_label="Yes"
)

f1_exp10 = f1_score(
    y_test_exp5,
    y_pred_exp10,
    pos_label="Yes"
)

roc_auc_exp10 = roc_auc_score(
    (y_test_exp5 == "Yes").astype(int),
    y_prob_exp10
)

In [ ]:
print("Precision:", precision_exp10)
print("Recall:", recall_exp10)
print("F1-Score:", f1_exp10)
print("ROC-AUC:", roc_auc_exp10)

Precision: 0.44074074074074077
Recall: 0.31989247311827956
F1-Score: 0.3707165109034268
ROC-AUC: 0.6629597370412998


In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test_exp5, y_pred_exp10))

Matriz de confusión:
[[886 151]
 [253 119]]
